# 04. TabPFN + EDA 시사점 적용

`03_eda`에서 찾은 시사점을 전처리 · 파생변수로 만들어서 **하나씩 넣어보고, CV가 좋아질 때만 채택**한다.

| 단계 | 내용 |
| --- | --- |
| 1 | 기준점: 02 베이스라인 전처리 (A) |
| 2 | 핵심 수정: `dir_prev_bfnum` 살리기 (B) → **타깃 변환** 비교 (원본 / log / sqrt) |
| 3 | 나머지 시사점 그룹을 하나씩 추가 (좋아질 때만 채택) |
| 4 | 원본 · 로그 타깃 예측 블렌딩 |
| 5 | 최종 학습 · 제출 파일 |

- 검증: 팀 공통 5-Fold (seed 42), **원래 관객수 단위 RMSE** (낮을수록 좋음)
- fold별 표준편차와 **흥행작(100만+)만의 RMSE**도 같이 봄 → 대박 영화를 잘 맞히는지 확인
- TabPFN은 CPU에서 1회 학습에 약 40초 → 실험 1개(5-fold)에 약 3~4분. 결과는 `cache_04/`에 저장돼서 **다시 실행하면 건너뜀**

## 0. 설정

In [1]:
import os
import re
import time
import pickle
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn import set_config
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from tabpfn import TabPFNRegressor

warnings.filterwarnings('ignore')
set_config(display='text')

DATA_DIR = '.'
SEED = 42
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CACHE_DIR = 'cache_04'
os.makedirs(CACHE_DIR, exist_ok=True)

train = pd.read_csv(os.path.join(DATA_DIR, 'movies_train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'movies_test.csv'))
submission = pd.read_csv(os.path.join(DATA_DIR, 'submission.csv'))
y = train['box_off_num']
print('device:', DEVICE, '| train', train.shape, '| test', test.shape)

device: cpu | train (600, 12) | test (243, 11)


C:\Users\taehe\OneDrive\문서\대학교\3학년 2학기\ESAA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 파생변수 함수
옵션(`opts`)으로 시사점 그룹을 켜고 끌 수 있게 만듦.

| 그룹 | EDA 근거 | 만드는 변수 |
| --- | --- | --- |
| `debut` | `dir_prev_bfnum` 결측 = 감독 데뷔작, 관객수 중앙값 3,401 vs 222,258 | 컬럼 삭제 대신 0으로 채우고 `is_debut` 플래그 |
| `log_num` | `num_staff`, `dir_prev_bfnum` 오른쪽으로 치우침 | `log1p` 변환 |
| `distributor` | 같은 회사 표기 중복 (CJ 4가지 등), 편수 적은 배급사 다수 | 이름 정규화 → 4대 배급사 · 편수 기반 묶기 · 배급사 편수 |
| `date` | 개봉 시기 효과 | 요일 · 분기 · 성수기(7·8·12·1월) 플래그 |
| `rating` | 15세가 흥행 중심, 전체 관람가는 규모가 커도 흥행 어려움 | 15세 · 전체 관람가 플래그, 등급 × log(스태프 수) |
| `title` | 숫자 포함 = 속편 가능성 | 숫자 포함 · 부제 포함 플래그 |

In [2]:
RATINGS = ['전체 관람가', '12세 관람가', '15세 관람가', '청소년 관람불가']
BIG4 = {'CJ': 'CJ', '롯데': '롯데', 'NEW': 'NEW', '쇼박스': '쇼박스'}


def norm_distributor(name):
    n = re.sub(r'\(주\)|㈜|주식회사|\s', '', name)
    for key, brand in BIG4.items():
        if key.lower() in n.lower():
            return brand
    return n


def make_features(train, test, opts):
    tr, te = train.copy(), test.copy()
    feats = []

    # ── 공통: 기본 수치형 + 개봉 연·월 (02 베이스라인과 동일)
    for df in (tr, te):
        d = pd.to_datetime(df['release_time'])
        df['년'] = d.dt.year
        df['월'] = d.dt.month
    feats += ['time', 'dir_prev_num', 'num_staff', 'num_actor', '년', '월']

    # ── debut: dir_prev_bfnum 살리기
    if opts.get('debut'):
        for df in (tr, te):
            df['is_debut'] = df['dir_prev_bfnum'].isna().astype(int)
            df['dir_prev_bfnum'] = df['dir_prev_bfnum'].fillna(0)
        feats += ['dir_prev_bfnum', 'is_debut']

    # ── log_num: 치우친 수치형 로그 변환 (원래 컬럼 대신 사용)
    if opts.get('log_num'):
        for col in ['num_staff', 'dir_prev_bfnum']:
            if col in feats:
                for df in (tr, te):
                    df[f'log_{col}'] = np.log1p(df[col])
                feats = [f'log_{col}' if f == col else f for f in feats]

    # ── 배급사
    if opts.get('distributor'):
        for df in (tr, te):
            df['dist_norm'] = df['distributor'].map(norm_distributor)
        cnt = tr['dist_norm'].value_counts()
        for df in (tr, te):
            df['dist_count'] = df['dist_norm'].map(cnt).fillna(0)
            df['is_big4'] = df['dist_norm'].isin(BIG4.values()).astype(int)
            df['dist_group'] = np.select(
                [df['dist_norm'].isin(BIG4.values()), df['dist_count'] >= 10, df['dist_count'] >= 2],
                [df['dist_norm'], '중견(10편+)', '소형(2~9편)'], default='1편·신규')
        feats += ['dist_count', 'is_big4', 'dist_group']
    else:  # 02 베이스라인: 상위 5개 + 기타
        top5 = tr['distributor'].value_counts().index[:5]
        for df in (tr, te):
            df['dist_group'] = df['distributor'].where(df['distributor'].isin(top5), '기타')
        feats += ['dist_group']

    # ── 개봉 시기
    if opts.get('date'):
        for df in (tr, te):
            d = pd.to_datetime(df['release_time'])
            df['weekday'] = d.dt.dayofweek
            df['quarter'] = d.dt.quarter
            df['is_peak'] = d.dt.month.isin([7, 8, 12, 1]).astype(int)
        feats += ['weekday', 'quarter', 'is_peak']

    # ── 관람 등급
    if opts.get('rating'):
        for df in (tr, te):
            df['is_15'] = (df['screening_rat'] == '15세 관람가').astype(int)
            df['is_all'] = (df['screening_rat'] == '전체 관람가').astype(int)
            log_staff = np.log1p(df['num_staff'])
            for i, r in enumerate(RATINGS):
                df[f'staff_x_rat{i}'] = log_staff * (df['screening_rat'] == r)
        feats += ['is_15', 'is_all'] + [f'staff_x_rat{i}' for i in range(len(RATINGS))]

    # ── 제목
    if opts.get('title'):
        for df in (tr, te):
            df['title_has_digit'] = df['title'].str.contains(r'\d').astype(int)
            df['title_has_sub'] = df['title'].str.contains(r'[:：\-]').astype(int)
        feats += ['title_has_digit', 'title_has_sub']

    # ── 범주형 원핫 (장르 · 등급 · 배급사 그룹)
    cat_cols = ['genre', 'screening_rat', 'dist_group']
    X = pd.get_dummies(tr[feats + cat_cols[:2]], columns=cat_cols, dtype=float)
    X_test = pd.get_dummies(te[feats + cat_cols[:2]], columns=cat_cols, dtype=float).reindex(columns=X.columns, fill_value=0)
    return X.astype(float), X_test.astype(float)


X_tmp, _ = make_features(train, test, {'debut': 1, 'log_num': 1, 'distributor': 1, 'date': 1, 'rating': 1, 'title': 1})
print('모든 그룹을 켰을 때 변수', X_tmp.shape[1], '개')
print(list(X_tmp.columns))

모든 그룹을 켰을 때 변수 44 개
['time', 'dir_prev_num', 'log_num_staff', 'num_actor', '년', '월', 'log_dir_prev_bfnum', 'is_debut', 'dist_count', 'is_big4', 'weekday', 'quarter', 'is_peak', 'is_15', 'is_all', 'staff_x_rat0', 'staff_x_rat1', 'staff_x_rat2', 'staff_x_rat3', 'title_has_digit', 'title_has_sub', 'genre_SF', 'genre_공포', 'genre_느와르', 'genre_다큐멘터리', 'genre_드라마', 'genre_멜로/로맨스', 'genre_뮤지컬', 'genre_미스터리', 'genre_서스펜스', 'genre_애니메이션', 'genre_액션', 'genre_코미디', 'screening_rat_12세 관람가', 'screening_rat_15세 관람가', 'screening_rat_전체 관람가', 'screening_rat_청소년 관람불가', 'dist_group_1편·신규', 'dist_group_CJ', 'dist_group_NEW', 'dist_group_롯데', 'dist_group_소형(2~9편)', 'dist_group_쇼박스', 'dist_group_중견(10편+)']


## 2. 팀 공통 CV 함수 (+ 결과 캐시)
- `target='raw' | 'log' | 'sqrt'`: 학습할 때만 변환하고, **예측은 원래 단위로 되돌려서** RMSE 계산
- 출력: 전체 RMSE · fold별 RMSE 표준편차 · 흥행작(100만+) RMSE

In [3]:
TRANSFORMS = {
    'raw':  (lambda v: v,          lambda p: p),
    'log':  (np.log1p,             np.expm1),
    'sqrt': (np.sqrt,              lambda p: np.square(np.clip(p, 0, None))),
}
RESULTS = []


def make_tabpfn():
    return TabPFNRegressor(device=DEVICE, random_state=SEED)


def run_cv(name, opts, target='raw'):
    path = os.path.join(CACHE_DIR, f'{name}.pkl')
    if os.path.exists(path):
        with open(path, 'rb') as f:
            res = pickle.load(f)
        print(f'[캐시] {name}')
    else:
        X, _ = make_features(train, test, opts)
        fwd, inv = TRANSFORMS[target]
        oof = np.zeros(len(y))
        fold_rmse = []
        t0 = time.time()
        for tr_idx, va_idx in kf.split(X):
            model = make_tabpfn()
            model.fit(X.iloc[tr_idx], fwd(y.iloc[tr_idx]))
            oof[va_idx] = np.clip(inv(model.predict(X.iloc[va_idx])), 0, None)
            fold_rmse.append(mean_squared_error(y.iloc[va_idx], oof[va_idx]) ** 0.5)
        hit = y >= 1_000_000
        res = {
            '실험': name, '타깃': target, '변수 수': X.shape[1],
            'CV RMSE': mean_squared_error(y, oof) ** 0.5,
            'fold 표준편차': np.std(fold_rmse),
            '흥행작 RMSE': mean_squared_error(y[hit], oof[hit]) ** 0.5,
            '소요(초)': time.time() - t0,
            'opts': opts, 'oof': oof,
        }
        with open(path, 'wb') as f:
            pickle.dump(res, f)
    RESULTS.append(res)
    print(f"{name:<28} CV RMSE {res['CV RMSE']:>12,.0f} | fold std {res['fold 표준편차']:>10,.0f} | 흥행작 RMSE {res['흥행작 RMSE']:>12,.0f}")
    return res


def show_results():
    df = pd.DataFrame(RESULTS).drop(columns=['opts', 'oof']).drop_duplicates('실험', keep='last')
    return df.style.format({'CV RMSE': '{:,.0f}', 'fold 표준편차': '{:,.0f}', '흥행작 RMSE': '{:,.0f}', '소요(초)': '{:.0f}'}) \
             .highlight_min(subset=['CV RMSE'], color='#c6efce')

## 3. 기준점 · 핵심 수정 · 타깃 변환

In [4]:
BASE = {}
res_A = run_cv('A_baseline', BASE, 'raw')                  # 02 베이스라인과 같은 조건

DEBUT = {'debut': 1}
res_B = {t: run_cv(f'B_debut_{t}', DEBUT, t) for t in ['raw', 'log', 'sqrt']}

best_target = min(res_B, key=lambda t: res_B[t]['CV RMSE'])
print('\n→ 채택할 타깃 변환:', best_target)

[캐시] A_baseline
A_baseline                   CV RMSE    1,385,455 | fold std    299,019 | 흥행작 RMSE    3,051,668
[캐시] B_debut_raw
B_debut_raw                  CV RMSE    1,388,439 | fold std    301,182 | 흥행작 RMSE    3,063,062
[캐시] B_debut_log
B_debut_log                  CV RMSE    1,455,090 | fold std    368,964 | 흥행작 RMSE    3,365,861


B_debut_sqrt                 CV RMSE    1,388,389 | fold std    321,576 | 흥행작 RMSE    3,129,809

→ 채택할 타깃 변환: sqrt


## 4. 나머지 시사점 하나씩 추가 (좋아질 때만 채택)
순서대로 그룹을 하나 켜보고, CV RMSE가 **현재 최고보다 낮아지면 채택**, 아니면 다시 끔.

In [5]:
current = dict(DEBUT)
best = res_B[best_target]
log_rows = []

for group in ['log_num', 'distributor', 'date', 'rating', 'title']:
    trial = {**current, group: 1}
    name = 'C_' + '+'.join(k for k in trial if trial[k]) + f'_{best_target}'
    res = run_cv(name, trial, best_target)
    improved = res['CV RMSE'] < best['CV RMSE']
    log_rows.append({'그룹': group, 'CV RMSE': res['CV RMSE'], '이전 최고': best['CV RMSE'],
                     '변화': res['CV RMSE'] - best['CV RMSE'], '채택': '✅' if improved else '❌'})
    if improved:
        current, best = trial, res

display(pd.DataFrame(log_rows).style.format({'CV RMSE': '{:,.0f}', '이전 최고': '{:,.0f}', '변화': '{:+,.0f}'}))
print('최종 채택 그룹:', [k for k in current if current[k]])

C_debut+log_num_sqrt         CV RMSE    1,391,572 | fold std    323,519 | 흥행작 RMSE    3,137,375


C_debut+distributor_sqrt     CV RMSE    1,385,455 | fold std    310,779 | 흥행작 RMSE    3,122,480


C_debut+distributor+date_sqrt CV RMSE    1,370,629 | fold std    330,228 | 흥행작 RMSE    3,086,862


C_debut+distributor+date+rating_sqrt CV RMSE    1,357,181 | fold std    320,882 | 흥행작 RMSE    3,051,397


C_debut+distributor+date+rating+title_sqrt CV RMSE    1,357,980 | fold std    313,843 | 흥행작 RMSE    3,052,499


,그룹,CV RMSE,이전 최고,변화,채택
0,log_num,"1,391,572","1,388,389","+3,183",❌
1,distributor,"1,385,455","1,388,389","-2,934",✅
2,date,"1,370,629","1,385,455","-14,826",✅
3,rating,"1,357,181","1,370,629","-13,448",✅
4,title,"1,357,980","1,357,181",+799,❌


최종 채택 그룹: ['debut', 'distributor', 'date', 'rating']


## 5. 원본 · 로그 타깃 블렌딩
원본 타깃은 대박 영화에 강하고, 로그 타깃은 작은 영화에 강함 → 두 예측을 섞으면 서로 보완될 수 있음.
가중치는 OOF 예측으로 정함 (같은 데이터로 고르는 거라 CV가 약간 낙관적일 수 있음 → 변화가 작으면 블렌딩 안 쓰기).

In [6]:
final_name = lambda t: 'D_final_' + '+'.join(k for k in current if current[k]) + f'_{t}'
# 이미 돌린 조합(현재 최고)은 다시 돌리지 않음
res_raw = best if best['타깃'] == 'raw' else run_cv(final_name('raw'), current, 'raw')
res_log = best if best['타깃'] == 'log' else run_cv(final_name('log'), current, 'log')

rows = []
for w in np.round(np.arange(0, 1.01, 0.1), 1):
    blend = w * res_raw['oof'] + (1 - w) * res_log['oof']
    rows.append({'원본 가중치 w': w, 'CV RMSE': mean_squared_error(y, blend) ** 0.5})
blend_df = pd.DataFrame(rows)
best_w = blend_df.loc[blend_df['CV RMSE'].idxmin(), '원본 가중치 w']
display(blend_df.style.format({'CV RMSE': '{:,.0f}'}).highlight_min(subset=['CV RMSE'], color='#c6efce'))

single_best = min(res_raw['CV RMSE'], res_log['CV RMSE'], best['CV RMSE'])
blend_best = blend_df['CV RMSE'].min()
USE_BLEND = blend_best < single_best * 0.99   # 1% 이상 좋아질 때만 블렌딩 사용
print(f'단일 모델 최고 {single_best:,.0f} vs 블렌딩 최고 {blend_best:,.0f} (w={best_w}) → 블렌딩 사용: {USE_BLEND}')

D_final_debut+distributor+date+rating_raw CV RMSE    1,366,317 | fold std    293,424 | 흥행작 RMSE    3,012,582


D_final_debut+distributor+date+rating_log CV RMSE    1,425,391 | fold std    360,537 | 흥행작 RMSE    3,290,102


,원본 가중치 w,CV RMSE
0,0.000000,"1,425,391"
1,0.100000,"1,412,695"
2,0.200000,"1,401,433"
3,0.300000,"1,391,642"
4,0.400000,"1,383,351"
5,0.500000,"1,376,589"
6,0.600000,"1,371,378"
7,0.700000,"1,367,735"
8,0.800000,"1,365,673"
9,0.900000,"1,365,200"


단일 모델 최고 1,357,181 vs 블렌딩 최고 1,365,200 (w=0.9) → 블렌딩 사용: False


## 6. 전체 실험 결과

In [7]:
display(show_results())
pd.DataFrame(RESULTS).drop(columns=['opts', 'oof']).drop_duplicates('실험', keep='last').to_csv('04_results.csv', index=False, encoding='utf-8-sig')
print('04_results.csv 저장 (노션 리더보드 기록용)')

,실험,타깃,변수 수,CV RMSE,fold 표준편차,흥행작 RMSE,소요(초)
0,A_baseline,raw,28,"1,385,455","299,019","3,051,668",220
1,B_debut_raw,raw,30,"1,388,439","301,182","3,063,062",199
2,B_debut_log,log,30,"1,455,090","368,964","3,365,861",124
3,B_debut_sqrt,sqrt,30,"1,388,389","321,576","3,129,809",104
4,C_debut+log_num_sqrt,sqrt,30,"1,391,572","323,519","3,137,375",138
5,C_debut+distributor_sqrt,sqrt,33,"1,385,455","310,779","3,122,480",265
6,C_debut+distributor+date_sqrt,sqrt,36,"1,370,629","330,228","3,086,862",156
7,C_debut+distributor+date+rating_sqrt,sqrt,42,"1,357,181","320,882","3,051,397",92
8,C_debut+distributor+date+rating+title_sqrt,sqrt,44,"1,357,980","313,843","3,052,499",274
9,D_final_debut+distributor+date+rating_raw,raw,42,"1,366,317","293,424","3,012,582",113


04_results.csv 저장 (노션 리더보드 기록용)


In [8]:
# 흥행작 예측이 얼마나 나아졌는지: 관객수 상위 15편의 실제값 vs 예측값
top = y.sort_values(ascending=False).index[:15]
final_oof = (best_w * res_raw['oof'] + (1 - best_w) * res_log['oof']) if USE_BLEND else best['oof']
cmp = pd.DataFrame({
    '제목': train.loc[top, 'title'],
    '실제': y[top],
    '02 베이스라인 예측': res_A['oof'][top],
    '04 최종 예측': final_oof[top],
})
display(cmp.style.format({'실제': '{:,.0f}', '02 베이스라인 예측': '{:,.0f}', '04 최종 예측': '{:,.0f}'}))

,제목,실제,02 베이스라인 예측,04 최종 예측
372,국제시장,"14,262,766","3,547,145","3,320,149"
362,도둑들,"12,983,841","6,618,400","7,932,343"
530,7번방의 선물,"12,811,435","4,604,174","4,418,661"
498,암살,"12,706,663","4,829,779","6,326,636"
460,"광해, 왕이 된 남자","12,323,595","3,096,786","2,170,778"
122,변호인,"11,374,879","2,923,176","2,386,779"
496,설국열차,"9,350,351","987,026","734,905"
101,관상,"9,135,806","4,139,423","3,845,089"
505,해적: 바다로 간 산적,"8,666,208","3,721,099","3,376,182"
476,수상한 그녀,"8,659,725","3,224,819","4,407,415"


## 7. 최종 학습 · 제출 파일

In [9]:
X, X_test = make_features(train, test, current)


def fit_predict(target):
    fwd, inv = TRANSFORMS[target]
    model = make_tabpfn()
    model.fit(X, fwd(y))
    return np.clip(inv(model.predict(X_test)), 0, None)


if USE_BLEND:
    pred = best_w * fit_predict('raw') + (1 - best_w) * fit_predict('log')
    desc = f'blend(w_raw={best_w})'
else:
    pred = fit_predict(best['타깃'])
    desc = best['타깃']

submission['box_off_num'] = pred
submission.to_csv('04_tabpfn_insights.csv', index=False)
print('저장: 04_tabpfn_insights.csv |', desc, '| 변수', X.shape[1], '개 |', [k for k in current if current[k]])
submission.head()

저장: 04_tabpfn_insights.csv | sqrt | 변수 42 개 | ['debut', 'distributor', 'date', 'rating']


,title,box_off_num
0,용서는 없다,1.290817e+06
1,아빠가 여자를 좋아해,1.912159e+06
2,하모니,1.996151e+06
3,의형제,2.371787e+06
4,평행 이론,1.551763e+06
